# Reading the Radar
### A hands-on guide to what weather radar sees — and misses

*Sebastián Torres · CIWRO, University of Oklahoma*  
*Clouds4Africa · July 2026*

---
**How to set up this notebook.** Wait until the top-right kernel indicator shows Python (Pyodide) and stops saying "Starting…" (the first load can take 30–60 seconds — this is normal). If the top-right says "No Kernel" instead, pick Kernel → Change Kernel → Python (Pyodide). Go to the Run menu → Run All Cells. Scroll up/down and interact with the sliders. If anything looks broken on the very first try, run that one cell again (Shift+Enter).

**How to use this notebook.** Connect what you see on a radar display to the physics and system design behind it. Each widget has three question levels — *Basic*, *A little further*, *Going deeper* — so start where you're comfortable and push as far as you can; they're meant to stretch, not to be finished. Time per widget is short, so don't expect to clear every level here; the more advanced questions are designed to be picked up again after the workshop. The notebook is yours to keep and re-run afterward.

<sub>© 2026 Sebastián Torres · Licensed under CC BY-NC 4.0 · Contact: sebas@ou.edu · Materials: https://sebastiantorr.github.io/Clouds4Africa/tree/</sub>

In [ ]:
%pip install ipywidgets

# Block 2: One measurement
### How big is a single radar value — and can the radar even detect it?

Block 1 showed *where* the beam goes. This block asks how big one measurement actually is, and whether the radar can detect it at all — a value is a chunk of air with a real size, set partly by the beam (Block 1) and partly by the pulse. This block has three widgets. Run the setup cell once, then work through the widgets in order, since each builds on the last.

## Before we start
This block builds directly on Block 1's beam geometry. Three ideas frame it.

1. **A radar value is an average over a volume, not a point.** Its depth along the beam is set by the *pulse* (≈ cτ/2); its width and height are set by the *beam* (≈ r·θ, from Block 1).

2. **Longer pulse, more energy.** Sensitivity grows with pulse length — but the sample gets deeper and the range resolution coarsens at the same time. Almost every choice here is a trade-off.

3. **Detection is not guaranteed.** Whether a faint echo is seen depends on range: received power falls as 1/r², so the weakest *detectable* reflectivity climbs with distance. "No echo" can mean "too weak to detect," not "no weather."

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider, Dropdown

C = 2.998e8                 # speed of light, m/s
kB, T0 = 1.380649e-23, 290.0

def beamwidth(freq_ghz=2.8, dish_m=8.5):
    # half-power beamwidth (deg)
    return np.degrees(1.27 * (C/(freq_ghz*1e9)) / dish_m)

def min_Z(r_km, power_kW=750, dish_m=8.5, freq_ghz=2.8, pulse_us=1.57,
          eta=0.45, NF_dB=4, Kw2=0.93, snr_dB=3):
    # minimum detectable reflectivity (dBZ) vs range, from the weather radar equation
    lam = C/(freq_ghz*1e9); tau = pulse_us*1e-6; r = np.asarray(r_km, float)*1e3
    G = eta*(np.pi*dish_m/lam)**2; theta = 1.27*lam/dish_m
    N = kB*T0*(1/tau)*10**(NF_dB/10); snr = 10**(snr_dB/10)
    K = (np.pi**3 * C * power_kW*1e3 * G**2 * theta**2 * tau * Kw2)/(1024*np.log(2)*lam**2)
    return 10*np.log10((snr*N*r**2/K)/1e-18)

print("Block 2 ready | range depth at 1.57 us:", round(C*1.57e-6/2), "m | MDS @100 km:", round(float(min_Z(100)),1), "dBZ")

## 1 — The pulse: one slider, three consequences

Pulse length τ pulls three ways at once: **range resolution** Δr = cτ/2, **sensitivity** (∝ τ — a longer pulse carries more energy), and the **blind range** near the radar (it can't listen while it's still transmitting). Two targets sit in front of the radar — lengthen the pulse and watch them blur together.

In [ ]:
def plot_pulse(pulse_us=1.5, separation_m=400):
    dr_m = C*pulse_us*1e-6/2; r = np.linspace(0, 4, 2000)
    sigma = (dr_m/1000)/2.355; t1, t2 = 2.0, 2.0 + separation_m/1000
    profile = np.exp(-0.5*((r-t1)/sigma)**2) + np.exp(-0.5*((r-t2)/sigma)**2)
    plt.figure(figsize=(8, 4.6))
    plt.axvspan(0, dr_m/1000, color="gray", alpha=0.2, label="blind range")
    plt.plot(r, profile, lw=2)
    plt.axvline(t1, color="k", ls=":", lw=0.8); plt.axvline(t2, color="k", ls=":", lw=0.8)
    plt.xlim(0, 4); plt.ylim(0, 2.1); plt.grid(alpha=0.3); plt.legend()
    plt.xlabel("range (km)"); plt.ylabel("relative power")
    plt.title("dr = " + str(round(dr_m)) + " m  |  blind range " + str(round(dr_m)) + " m  |  sensitivity "
              + format(10*np.log10(pulse_us/1.5), "+.1f") + " dB vs short pulse")
    plt.show()

interact(plot_pulse,
         pulse_us=FloatSlider(value=1.5, min=0.5, max=7, step=0.1, description="pulse µs"),
         separation_m=FloatSlider(value=400, min=50, max=1500, step=50, description="sep m"));

**Basic**

1. With the two targets 400 m apart, are they resolved as two peaks at the short pulse (1.5 µs)? Lengthen the pulse until they merge into one — what is Δr there, compared to 400 m?
2. Read the blind range near the radar at 1.5 µs and at 4.5 µs. Which pulse lets you see closer to the radar?

**A little further**

1. By how many dB does sensitivity change going from 1.5 to 4.5 µs? Which pulse would you choose to hunt weak clear-air echo — and what do you give up to get it?
2. Two equal targets just separate when their spacing ≈ Δr. Test this: find the merge separation at two different pulse lengths and check that it tracks cτ/2.

**Going deeper**

1. Sensitivity grows with pulse length while resolution Δr = cτ/2 coarsens with it — opposite directions. Set the pulse so Δr ≈ 1 km and read the sensitivity gain in dB over the 1.5 µs pulse. State the rule per doubling of τ for each (resolution and sensitivity), and explain why any single pulse length is a compromise.
2. At 4.5 µs the first ~700 m are blind. A shallow boundary-layer is 500 m from the radar — can a long-pulse clear-air scan see it at all? Why does clear-air mode accept such a large blind zone anyway?

## 2 — The whole sample: the resolution volume

Combine the beam (Block 1) with the pulse: one measurement is approximately "a box" ≈ r·θ wide **and** tall (it grows with range) by Δr = cτ/2 deep (fixed by the pulse). Far from the radar, that's a surprisingly large chunk of air to average into a single number.

In [ ]:
def plot_volume(range_km=120, pulse_us=1.5):
    bw = beamwidth()
    width = range_km*np.radians(bw); depth = C*pulse_us*1e-6/2/1000
    volume = depth*np.pi*(width/2)**2; r = np.linspace(1, 230, 300)
    plt.figure(figsize=(8, 4.5))
    plt.plot(r, r*np.radians(bw)*1000, lw=2, label="across-beam = vertical (r·θ)")
    plt.axhline(C*pulse_us*1e-6/2, color="tab:orange", lw=2, label="range depth (cτ/2)")
    plt.axvline(range_km, color="tab:red", ls=":")
    plt.ylim(0, 4000); plt.grid(alpha=0.3); plt.legend()
    plt.xlabel("range (km)"); plt.ylabel("dimension (m)")
    plt.title("at " + str(range_km) + " km: " + str(round(width,1)) + " km wide/tall x "
              + str(round(depth*1000)) + " m deep ~ " + str(round(volume,2)) + " km³")
    plt.show()

interact(plot_volume,
         range_km=IntSlider(value=120, min=10, max=230, step=10, description="range km"),
         pulse_us=FloatSlider(value=1.5, min=0.5, max=5, step=0.1, description="pulse µs"));

**Basic**

1. How wide and tall is the sample at 60 km versus 230 km?
2. The radial depth stays ~250 m at every range. Which slider changes it, and which one leaves it alone?

**A little further**

1. A convective core is ~2 km across. Beyond what range does the beam grow wider than the core, so the peak reflectivity starts reading low (beam filling)?
2. Roughly how many km³ of air does one pixel average at 150 km versus 230 km? Why does that matter for rainfall estimates and echo-top heights?

**Going deeper**

1. Both cross-beam dimensions grow with range, so the sample volume grows as r². Read the volume at 60 km and at 230 km and confirm the ratio matches (230/60)². What important consequence does this have for radar observations at long range?
2. The beam's vertical extent is r·θ and grows with range. A bright band is a ~500 m thick layer. Find the range beyond which the beam's vertical extent exceeds 500 m, so the bright band can no longer be vertically resolved — and explain why bright-band contamination is a sharp ring near the radar but a broad, smeared bias far out.

## 3 — Can you even detect it? The radar equation

Received power scales as Pt·D²·τ·Z / (λ⁴ r²), so the faintest echo the radar can detect — the minimum detectable Z — climbs with range. Slide the hardware and watch the whole sensitivity curve move; drop a target on the plot and see whether it clears the noise.

In [ ]:
def plot_radar_eq(power_kW=750, dish_m=8.5, freq_ghz=2.8, pulse_us=1.57, target_dBZ=10, target_km=150):
    r = np.linspace(10, 460, 400); zmin = min_Z(r, power_kW, dish_m, freq_ghz, pulse_us)
    bw = np.degrees(1.27*(C/(freq_ghz*1e9))/dish_m)
    detect = target_dBZ >= np.interp(target_km, r, zmin)
    plt.figure(figsize=(8, 4.6))
    plt.plot(r, zmin, "k", lw=2, label="min detectable Z")
    plt.fill_between(r, zmin, -30, alpha=0.05)
    for z, lab in [(10, "drizzle"), (20, "light rain"), (45, "heavy rain")]:
        plt.axhline(z, color="gray", ls=":", lw=0.6); plt.text(455, z+0.6, lab, fontsize=7, ha="right", color="gray")
    plt.plot([target_km], [target_dBZ], "o", ms=11, color="tab:green" if detect else "tab:red")
    plt.annotate("detected" if detect else "below noise", (target_km, target_dBZ), (target_km+8, target_dBZ+5), fontsize=9)
    plt.ylim(-30, 55); plt.grid(alpha=0.3); plt.legend(loc="lower right")
    plt.xlabel("range (km)"); plt.ylabel("reflectivity (dBZ)")
    plt.title("beamwidth " + str(round(bw,2)) + "°  |  MDS @100 km = "
              + str(round(float(min_Z(100, power_kW, dish_m, freq_ghz, pulse_us)),1)) + " dBZ")
    plt.show()

interact(plot_radar_eq,
         power_kW=FloatSlider(value=750, min=100, max=1500, step=50, description="power kW"),
         dish_m=FloatSlider(value=8.5, min=2, max=15, step=0.5, description="dish m"),
         freq_ghz=FloatSlider(value=2.8, min=2, max=10, step=0.2, description="freq GHz"),
         pulse_us=FloatSlider(value=1.5, min=0.5, max=5, step=0.1, description="pulse µs"),
         target_dBZ=FloatSlider(value=10, min=-20, max=50, step=5, description="target dBZ"),
         target_km=FloatSlider(value=150, min=20, max=450, step=10, description="target km"));

**Basic**

1. At NEXRAD defaults (Pt = 750 kW, D = 8.5 m, f = 2.8 GHz, τ = 1.5 µs), read the minimum detectable Z at 100 km. Now double the transmit power — how many dBZ does it improve?
2. Put a +10 dBZ echo (light rain) at 450 km. Is it detected? Which single slider rescues it most easily?

**A little further**

1. Keep the dish at D = 8.5 m and move from S-band (2.8 GHz) to X-band (9.4 GHz). How many dB does the minimum detectable Z change, and how does that match the 1/λ⁴ scaling?
2. Now shrink the dish *as* you shorten the wavelength. Set the dish to 2 m at X-band
   (9.4 GHz) and read the minimum detectable Z at 100 km; compare it to NEXRAD's (8.5 m
   dish, S-band). Can a dish a quarter the size keep up with — or even beat — NEXRAD?
   That's how a compact X-band radar stays sensitive. The price of going to
   such a short wavelength is waiting for you in Block 4.

**Going deeper**

1. This curve (min-detectable-Z ∝ r², i.e. +6 dB per range doubling) is the *beam-filled* law. A point target — a bird or a plane — instead falls off as 1/r⁴. Explain why distributed weather falls only as r² while a point target falls as r⁴ (hint: the resolution volume grows as r² and fills with scatterers). Which falloff is gentler, and what does that mean for detecting weak distant rain?
2. Combine all three of this block's ideas: lengthening the pulse raises sensitivity (this widget), coarsens resolution (Widget 1), and deepens the sample (Widget 2). Pick a pulse length that just detects 0 dBZ at 200 km, then state the resolution and sample-depth price you paid. Is that a good pulse for severe convection, or for clear-air mode?

## 4 — Radar data: one storm, two ranges

The same storm at the same time — the 20 May 2013 Moore, OK supercell — from two radars,
both at 0.48° and centred on the storm: KTLX (Twin Lakes, ~20 km, left, 20:16) and KINX
(Tulsa, ~200 km, right, 20:17). (Ignore the widespread weak speckle near KTLX — much of it
is clear-air return and clutter that the close, sensitive radar picks up.)

![Image](https://raw.githubusercontent.com/sebastiantorr/Clouds4Africa/main/content/images/block2_image1.png)

**Basic**

1. Which panel catches the faint, widespread echo around the storm, and which keeps only the
   strong cores? That contrast is the near radar versus the far one — which is which, and why?
2. Now the cores: one panel resolves tighter, finer structure; the other smears the same storm
   into coarser blobs. Which is the near radar, and which idea from this block explains the
   difference?

**A little further**

1. From the radar-equation widget, the far radar is ~10× the range, so its minimum detectable Z
   sits about +20 dB higher (min-Z ∝ r²). Connect that number to the weak periphery that
   vanishes in the far panel — roughly what echo strength drops below its noise floor?
2. The far radar's 0.48° beam at 200 km is also ~4 km up, so part of why its panel looks
   different is height, not resolution. Can beam height alone make an echo *broader*? If not,
   what must the broadening be doing on its own (beam ≈ 0.3 km at KTLX vs ~3.3 km at KINX)?

**Going deeper**

1. Remove the height confound (second image below). KINX's 0.5° at 200 km is ~4 km up; the KTLX tilt
   that also reaches ~4 km at ~20 km is high — around 12° (the panel shows 12.48°). With both
   sampling about the same height, the only thing left is beam width: does the far storm now look
   clearly broader and coarser? Note too that the near 12.48° panel is a compact disc — explain
   why a high tilt loses the storm with range.
2. Estimate the sample-volume ratio: both cross-beam dimensions scale with range and the depth
   cτ/2 ≈ 0.25 km is shared, so volume ∝ r². At ~10× the range, one KINX pixel averages ~100×
   more air — what does smearing the storm across that much atmosphere do to its peak?

![Image](https://raw.githubusercontent.com/sebastiantorr/Clouds4Africa/main/content/images/block2_image2.png)

## Block 2 takeaway
**One radar value is a volume-sized average whose size and detectability you set by trading resolution, sensitivity, and range against each other — so a distant pixel is a large spatial average, a possibly-underestimated peak, and a detection threshold, all at once.**